# 🤖 Microsoft Foundry Prompt Agents Quickstart

This notebook demonstrates how to create and interact with **Prompt Agents** using the
**Azure AI Projects SDK (v2.x)** and the **Foundry Agent Service**.

A **Prompt Agent** is a declaratively defined agent that combines:
- A deployed model
- System instructions
- Optional tools (MCP, Web Search, Code Interpreter, File Search)
- Natural language prompts to drive behavior

Best for: Rapid prototyping, internal tools, and agents that don't need custom orchestration logic. Create a working agent in minutes using the portal or using the SDK.

> Reference: [Quickstart: Create a prompt agent – Microsoft Learn](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/prompt-agent?tabs=python)


In [1]:
# %pip install --quiet "azure-ai-projects>=2.0.0" azure-identity python-dotenv

In [2]:
import datetime
import json
import os
import sys
import textwrap

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pathlib import Path
from typing import Any, Optional, Sequence

In [3]:
print(f"Python  : {sys.version}")
print(f"Today   : {datetime.datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")

Python  : 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
Today   : 27-May-2026 08:13:16


## Configuration

Set your **Foundry project endpoint** and the name of your deployed model.

The endpoint has the format:
`https://<resource_name>.ai.azure.com/api/projects/<project_name>`

In [4]:
load_dotenv("azure.env")

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")

In [5]:
AGENT_NAME: str = "mslearn-agent"
AGENT_MODEL: str = "gpt-5.4"

print(f"Agent name : {AGENT_NAME}")
print(f"Model      : {AGENT_MODEL}")

Agent name : mslearn-agent
Model      : gpt-5.4


## Create the Foundry Client

`DefaultAzureCredential` automatically picks up credentials from:
- Azure CLI (`az login`)
- Managed Identity (when running on Azure)
- Environment variables (`AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`, `AZURE_TENANT_ID`)


In [6]:
def create_project_client(endpoint: str) -> AIProjectClient:
    """Instantiate an AIProjectClient using keyless Entra ID authentication.

    Parameters
    ----------
    endpoint : str
        The Foundry project endpoint URL.

    Returns
    -------
    AIProjectClient
        An authenticated client ready to interact with the Foundry API.

    Raises
    ------
    ValueError
        If the endpoint string is empty.
    """
    if not endpoint:
        raise ValueError("PROJECT_ENDPOINT must not be empty.")

    credential = DefaultAzureCredential()
    return AIProjectClient(endpoint=endpoint, credential=credential)


In [7]:
try:
    project_client = create_project_client(PROJECT_ENDPOINT)
    print("Client created successfully.")
except Exception as exc:
    print(f"Failed to create client: {exc}")
    raise


Client created successfully.


## Create a Prompt Agent

A **Prompt Agent** is defined with:
- A **model** — any model deployed in your Foundry project
- **Instructions** — the system prompt that drives the agent's behaviour
- **Tools** — optional tools (MCP, Web Search, Code Interpreter, etc.)

Each call to `create_version` creates a new *version* of the named agent.


In [8]:
AGENT_INSTRUCTIONS: str = """\
You are an expert assistant specializing in Azure cloud services and AI.

For every factual question about Azure, Microsoft Foundry, .NET, M365, or
Power Platform, you MUST call the Microsoft Learn MCP tools to retrieve
authoritative documentation before answering. Never answer purely from your
own knowledge for product facts.

Cite the Microsoft Learn URLs you used at the end of each answer under a
'Sources' heading. If the documentation does not contain the answer, say so
explicitly.

Be concise, accurate, and structured. Use bullet points when listing items.
"""


In [9]:
def create_prompt_agent(
    client: AIProjectClient,
    agent_name: str,
    model: str,
    instructions: str,
    tools: Optional[Sequence[object]] = None,
) -> object:
    """Create a new version of a named prompt agent in the Foundry Agent Service.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The unique name for this agent within the project.
    model : str
        The deployed model identifier (e.g. 'gpt-4.1-mini').
    instructions : str
        The system-level instructions that define the agent's behaviour.
    tools : Optional[Sequence[object]]
        Tools to attach (MCPTool, WebSearchTool, CodeInterpreterTool,
        FileSearchTool, etc.). When None, the agent has no tools.

    Returns
    -------
    object
        The newly created agent version object.

    Raises
    ------
    RuntimeError
        If the Foundry API call fails.
    """
    definition_kwargs: dict[str, object] = {
        "model": model,
        "instructions": instructions,
    }
    if tools:
        definition_kwargs["tools"] = list(tools)

    try:
        return client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(**definition_kwargs),
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to create agent '{agent_name}': {exc}") from exc


In [10]:
mslearn_tool = MCPTool(
    server_label="microsoft_learn",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    allowed_tools=[
        "microsoft_docs_search",
        "microsoft_docs_fetch",
        "microsoft_code_sample_search",
    ],
)


In [11]:
try:
    agent = create_prompt_agent(
        client=project_client,
        agent_name=AGENT_NAME,
        model=AGENT_MODEL,
        instructions=AGENT_INSTRUCTIONS,
        tools=[mslearn_tool],
    )
    print(f"Created {agent.name} v{agent.version} with MS Learn MCP attached.")
except RuntimeError as exc:
    print(exc)

Created mslearn-agent v1 with MS Learn MCP attached.


## Single-Turn Chat with the Agent

Send a single question and retrieve the agent's response.
Token usage is captured and displayed for cost monitoring.


In [12]:
def chat_single_turn(
    client: AIProjectClient,
    agent_name: str,
    user_message: str,
) -> tuple[str, dict[str, int]]:
    """Send a single message to a Foundry prompt agent and return the response.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the target agent.
    user_message : str
        The user's input message.

    Returns
    -------
    tuple[str, dict[str, int]]
        A tuple of (response_text, usage_dict).
        usage_dict contains 'input_tokens', 'output_tokens', 'total_tokens'.

    Raises
    ------
    RuntimeError
        If the API call fails.
    """
    openai_client = client.get_openai_client()

    try:
        response = openai_client.responses.create(
            extra_body={
                "agent_reference": {
                    "name": agent_name,
                    "type": "agent_reference",
                }
            },
            input=user_message,
        )
    except Exception as exc:
        raise RuntimeError(f"API call failed: {exc}") from exc

    usage: dict[str, int] = {}
    if hasattr(response, "usage") and response.usage:
        usage = {
            "input_tokens": getattr(response.usage, "input_tokens", 0),
            "output_tokens": getattr(response.usage, "output_tokens", 0),
            "total_tokens": getattr(response.usage, "total_tokens", 0),
        }

    return response.output_text, usage

In [13]:
question: str = "What is Microsoft Foundry?"

try:
    answer, usage = chat_single_turn(project_client, AGENT_NAME, question)
    
    display(Markdown(f"**User:** {question}\n\n**Agent:** {answer}"))
    if usage:
        print(f"\nToken usage: {usage}")
except RuntimeError as exc:
    print(exc)


**User:** What is Microsoft Foundry?

**Agent:** Microsoft Foundry is Microsoft’s unified Azure platform-as-a-service for building, operating, and governing enterprise AI applications.

In practical terms, it brings together:

- Models
- Agents
- Tools
- Observability and evaluations
- Enterprise controls like RBAC, networking, and policy management

Key points from Microsoft Learn:

- It is designed for:
  - Application developers
  - ML engineers and data scientists
  - IT admins and platform engineers
- It provides:
  - A single Foundry resource with projects
  - A portal at `ai.azure.com`
  - APIs and SDKs for Python, C#, JavaScript/TypeScript (preview), and Java (preview)
- It supports:
  - Building agents with tools and memory
  - Deploying and managing models
  - Monitoring AI assets
  - Governance and security at enterprise scale

Microsoft also states that Foundry consolidates earlier Azure AI experiences, including Azure AI Studio / Azure AI Foundry, into the newer Microsoft Foundry experience.

Sources

- https://learn.microsoft.com/azure/foundry/what-is-foundry
- https://learn.microsoft.com/azure/foundry/concepts/architecture


Token usage: {'input_tokens': 14370, 'output_tokens': 314, 'total_tokens': 14684}


In [14]:
print(answer)

Microsoft Foundry is Microsoft’s unified Azure platform-as-a-service for building, operating, and governing enterprise AI applications.

In practical terms, it brings together:

- Models
- Agents
- Tools
- Observability and evaluations
- Enterprise controls like RBAC, networking, and policy management

Key points from Microsoft Learn:

- It is designed for:
  - Application developers
  - ML engineers and data scientists
  - IT admins and platform engineers
- It provides:
  - A single Foundry resource with projects
  - A portal at `ai.azure.com`
  - APIs and SDKs for Python, C#, JavaScript/TypeScript (preview), and Java (preview)
- It supports:
  - Building agents with tools and memory
  - Deploying and managing models
  - Monitoring AI assets
  - Governance and security at enterprise scale

Microsoft also states that Foundry consolidates earlier Azure AI experiences, including Azure AI Studio / Azure AI Foundry, into the newer Microsoft Foundry experience.

Sources

- https://learn.mic

In [15]:
print(usage)

{'input_tokens': 14370, 'output_tokens': 314, 'total_tokens': 14684}


## Streaming Single-Turn Chat

Uses the streaming API so the response is printed incrementally.
This avoids long silent waits when MCP tool calls are involved.


In [16]:
def chat_single_turn_streaming(
    client: AIProjectClient,
    agent_name: str,
    user_message: str,
    model: str = AGENT_MODEL,
) -> str:
    """Send a single message to a Foundry prompt agent using streaming output.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the target agent.
    user_message : str
        The user's input message.
    model : str
        The deployed model identifier. Required by the streaming endpoint.

    Returns
    -------
    str
        The full accumulated response text.

    Raises
    ------
    RuntimeError
        If the streaming API call fails.
    """
    openai_client = client.get_openai_client()

    print(f"User  : {user_message}")
    print("Agent : ", end="", flush=True)

    full_text: str = ""

    try:
        with openai_client.responses.stream(
            model=model,
            extra_body={
                "agent_reference": {
                    "name": agent_name,
                    "type": "agent_reference",
                }
            },
            input=user_message,
        ) as stream:
            for event in stream:
                # Debug: uncomment to inspect event structure if output is still empty
                # print(f"\n[DEBUG] event.type={getattr(event, 'type', '?')} | {event}", flush=True)
                if getattr(event, "type", None) == "response.output_text.delta":
                    delta: str = getattr(event, "delta", "") or ""
                    print(delta, end="", flush=True)
                    full_text += delta
    except Exception as exc:
        raise RuntimeError(f"Streaming API call failed: {exc}") from exc

    print()
    return full_text

In [17]:
try:
    streamed_answer = chat_single_turn_streaming(
        project_client,
        AGENT_NAME,
        "What is Claude Opus?",
        model=AGENT_MODEL,
    )
except RuntimeError as exc:
    print(exc)


User  : What is Claude Opus?
Agent : Claude Opus is Anthropic’s high-end Claude model family available in Microsoft Foundry. In Microsoft’s documentation, Claude Opus models are described as suited for:

- Complex reasoning
- Advanced code generation, analysis, and debugging
- Multimodal tasks, including image analysis

In Microsoft Foundry, the currently documented Claude Opus variants include:

- `claude-opus-4-7` (preview)
- `claude-opus-4-6` (preview)
- `claude-opus-4-5` (preview)
- `claude-opus-4-1` (preview)

Microsoft Learn also notes these broader Claude capabilities for the Claude family in Foundry:

- Adaptive thinking
- Extended thinking
- Image and text input
- Code generation

If you want the shortest practical definition: Claude Opus is Anthropic’s premium Claude model tier for harder reasoning and coding workloads, exposed in Microsoft Foundry as preview partner models.

Sources

- https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-claude


## Multi-Turn Conversation

Use a **conversation** object to maintain chat history across multiple turns.
The agent automatically tracks context from previous messages within the same conversation.

> **Key concept:** A `conversation` is a server-side session pass its `id`
> on every subsequent request to preserve history.

Token usage is tracked and accumulated per turn.


In [18]:
def run_multi_turn_conversation(
    client: AIProjectClient,
    agent_name: str,
    messages: list[str],
) -> list[dict[str, Any]]:
    """Run a multi-turn conversation with a Foundry prompt agent.

    A single conversation context is maintained across all turns,
    allowing the agent to reference previous exchanges.
    Token usage is accumulated and returned per turn.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the target agent.
    messages : list[str]
        Ordered list of user messages to send sequentially.

    Returns
    -------
    list[dict[str, Any]]
        A list of turn dicts with keys: 'turn', 'user', 'assistant', 'usage'.

    Raises
    ------
    RuntimeError
        If the conversation creation or any turn API call fails.
    """
    openai_client = client.get_openai_client()

    try:
        conversation = openai_client.conversations.create()
    except Exception as exc:
        raise RuntimeError(f"Failed to create conversation: {exc}") from exc

    print(f"Conversation ID: {conversation.id}")
    print("-" * 60)

    agent_ref: dict[str, Any] = {
        "agent_reference": {
            "name": agent_name,
            "type": "agent_reference",
        }
    }

    history: list[dict[str, Any]] = []
    total_tokens: int = 0

    for turn_index, user_message in enumerate(messages, start=1):
        try:
            response = openai_client.responses.create(
                conversation=conversation.id,
                extra_body=agent_ref,
                input=user_message,
            )
        except Exception as exc:
            raise RuntimeError(f"Turn {turn_index} failed: {exc}") from exc

        assistant_reply: str = response.output_text

        usage: dict[str, int] = {}
        if hasattr(response, "usage") and response.usage:
            usage = {
                "input_tokens": getattr(response.usage, "input_tokens", 0),
                "output_tokens": getattr(response.usage, "output_tokens", 0),
                "total_tokens": getattr(response.usage, "total_tokens", 0),
            }
            total_tokens += usage.get("total_tokens", 0)

        display(Markdown(
            f"**Turn {turn_index}**\n\n"
            f"**User:** {user_message}\n\n"
            f"**Agent:** {assistant_reply}"
        ))
        if usage:
            print(f"  Token usage this turn: {usage}")
        print()

        history.append({
            "turn": turn_index,
            "user": user_message,
            "assistant": assistant_reply,
            "usage": usage,
        })

    print(f"Total tokens consumed across all turns: {total_tokens}")
    return history


In [19]:
conversation_turns: list[str] = [
    "What is AutoML for Images?",
    "What are the models for object detection?",
    "Display some python code to build an object detection model",
    "Can I add extra models from HuggingFace?",
]

try:
    conversation_history = run_multi_turn_conversation(
        client=project_client,
        agent_name=AGENT_NAME,
        messages=conversation_turns,
    )
except RuntimeError as exc:
    print(exc)


Conversation ID: conv_3ccfcdbc4a2ad13b00pdFkb0CTewgPAxyznhLbp48sWTjlna6f
------------------------------------------------------------


**Turn 1**

**User:** What is AutoML for Images?

**Agent:** AutoML for Images is the **computer vision capability of automated machine learning in Azure Machine Learning**.

- It helps you **train and tune image models automatically** using labeled image data.
- Azure Machine Learning tries different algorithms and parameter combinations, then selects the model that best fits the metric you specify.
- You can create these models with the **Azure Machine Learning Python SDK**, and view jobs, models, and outputs in **Azure Machine Learning studio**.

AutoML for Images supports these tasks:

- **Multi-class image classification**
  - One label per image
  - Example: cat *or* dog *or* duck
- **Multi-label image classification**
  - Multiple labels per image
  - Example: cat *and* dog
- **Object detection**
  - Finds objects and draws **bounding boxes**
- **Instance segmentation**
  - Identifies objects at the **pixel level** with polygons

According to Microsoft Learn, this capability can also:

- Integrate with **Azure Machine Learning data labeling**
- Use labeled data to generate image models
- Let you optimize performance by choosing algorithms and tuning hyperparameters
- Let you **download or deploy** the resulting model as a web service in Azure Machine Learning
- Support operationalization with **MLOps** and **ML Pipelines**

If you want, I can also explain:
- how AutoML for Images works end-to-end,
- what data format it expects,
- or how it compares with Azure AI Vision / custom vision options.

## Sources
- https://learn.microsoft.com/azure/machine-learning/concept-automated-ml?view=azureml-api-2


  Token usage this turn: {'input_tokens': 16177, 'output_tokens': 423, 'total_tokens': 16600}



**Turn 2**

**User:** What are the models for object detection?

**Agent:** For **AutoML for Images object detection** in Azure Machine Learning, Microsoft Learn lists these supported **legacy model architectures**:

- **YOLOv5** — `yolov5`
- **Faster R-CNN ResNet FPN**
  - `fasterrcnn_resnet18_fpn`
  - `fasterrcnn_resnet34_fpn`
  - `fasterrcnn_resnet50_fpn`
  - `fasterrcnn_resnet101_fpn`
  - `fasterrcnn_resnet152_fpn`
- **RetinaNet ResNet FPN**
  - `retinanet_resnet50_fpn`

Microsoft Learn also says AutoML supports object detection models from **MMDetection 3.1.0**, and highlights these curated models:

- **Sparse R-CNN**
  - `mmd-3x-sparse-rcnn_r50_fpn_300-proposals_crop-ms-480-800-3x_coco`
  - `mmd-3x-sparse-rcnn_r101_fpn_300-proposals_crop-ms-480-800-3x_coco`
- **Deformable DETR**
  - `mmd-3x-deformable-detr_refine_twostage_r50_16xb2-50e_coco`
- **VFNet**
  - `mmd-3x-vfnet_r50-mdconv-c3-c5_fpn_ms-2x_coco`
  - `mmd-3x-vfnet_x101-64x4d-mdconv-c3-c5_fpn_ms-2x_coco`
- **YOLOF**
  - `mmd-3x-yolof_r50_c5_8x8_1x_coco`

A few useful notes from the docs:

- The default legacy object detection model is **`yolov5`**.
- AutoML can also use **any object detection model from the MMDetection 3.1.0 Model Zoo**, not just the curated list.
- For small object detection with tiling, Microsoft says it is supported for **all models supported by AutoML for Images for object detection**.

## Sources
- https://learn.microsoft.com/azure/machine-learning/how-to-auto-train-image-models?view=azureml-api-2
- https://learn.microsoft.com/azure/machine-learning/how-to-use-automl-small-object-detect?view=azureml-api-2

  Token usage this turn: {'input_tokens': 45257, 'output_tokens': 603, 'total_tokens': 45860}



**Turn 3**

**User:** Display some python code to build an object detection model

**Agent:** Here is a concise **Python example** from the Azure Machine Learning AutoML pattern to build an **object detection model**:

```python
from azure.ai.ml import MLClient, Input, automl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.automl import ObjectDetectionPrimaryMetrics
from azure.ai.ml.entities import AmlCompute
from azure.identity import DefaultAzureCredential

# Authenticate
credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id="<subscription-id>",
    resource_group_name="<resource-group>",
    workspace_name="<workspace-name>",
)

# Optional: create GPU compute cluster
compute_name = "gpu-cluster"
cluster_basic = AmlCompute(
    name=compute_name,
    type="amlcompute",
    size="Standard_NC64as_T4_v3",
    min_instances=0,
    max_instances=4,
    idle_time_before_scale_down=120,
)
ml_client.begin_create_or_update(cluster_basic)

# Training and validation data as MLTable inputs
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="./training-mltable-folder"
)

my_validation_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="./validation-mltable-folder"
)

# Experiment name
exp_name = "image-object-detection-experiment"

# Create AutoML object detection job
image_object_detection_job = automl.image_object_detection(
    compute=compute_name,
    experiment_name=exp_name,
    training_data=my_training_data_input,
    validation_data=my_validation_data_input,
    target_column_name="label",
    primary_metric=ObjectDetectionPrimaryMetrics.MEAN_AVERAGE_PRECISION,
    tags={"project": "object-detection-demo"},
)

# AutoMode limits
image_object_detection_job.set_limits(
    max_trials=10,
    max_concurrent_trials=2,
)

# Submit the job
returned_job = ml_client.jobs.create_or_update(image_object_detection_job)
print(f"Created job: {returned_job.name}")
```

Notes:
- Your input data must be in **MLTable** format, built from labeled image data in **JSONL** format.
- For image object detection, Microsoft Learn uses:
  - `automl.image_object_detection(...)`
  - `target_column_name="label"`
  - `primary_metric=ObjectDetectionPrimaryMetrics.MEAN_AVERAGE_PRECISION`
- AutoML for image tasks typically requires **GPU compute**.

If you want, I can also show:
- a **full working notebook version**,
- the **JSONL/MLTable format** for object detection data,
- or code for a **manual hyperparameter sweep** with YOLOv5.

## Sources
- https://learn.microsoft.com/azure/machine-learning/tutorial-auto-train-image-models?view=azureml-api-2

  Token usage this turn: {'input_tokens': 67026, 'output_tokens': 676, 'total_tokens': 67702}



**Turn 4**

**User:** Can I add extra models from HuggingFace?

**Agent:** Yes, **but with an important distinction**:

- For **AutoML for Images**, Microsoft Learn says you can use:
  - **any image classification model from the Hugging Face Hub** that is part of the **transformers** library
  - **any object detection or instance segmentation model from the MMDetection 3.1.0 Model Zoo**
- For **object detection specifically**, the docs mention **MMDetection models**, not Hugging Face object detection models.

So for your earlier **object detection** scenario:

- **Yes** to adding extra models if they are from **MMDetection 3.1.0**
- **No Microsoft Learn documentation was found saying AutoML image object detection supports arbitrary Hugging Face object detection models**

Also noted by Microsoft:

- Using any **HuggingFace or MMDetection** model causes runs to use **pipeline components**
- If you mix **legacy models** and **HuggingFace/MMDetection** models, all trials use components
- Azure ML also provides a **curated registry list** of tested models, but it is not limited to only those curated entries

If you meant **image classification** rather than object detection, then yes—Microsoft explicitly says you can use **any Hugging Face image classification model** from the Transformers library.

## Sources
- https://learn.microsoft.com/azure/machine-learning/how-to-auto-train-image-models?view=azureml-api-2

  Token usage this turn: {'input_tokens': 94852, 'output_tokens': 388, 'total_tokens': 95240}

Total tokens consumed across all turns: 225402


## Export Conversation History

Save the conversation to disk in JSON and Markdown formats.


In [20]:
def export_conversation(
    history: list[dict[str, Any]],
    output_dir: str = ".",
    filename_stem: str = "conversation",
) -> tuple[Path, Path]:
    """Export a conversation history to JSON and Markdown files.

    Parameters
    ----------
    history : list[dict[str, Any]]
        Output from `run_multi_turn_conversation`.
    output_dir : str
        Directory where output files are written. Created if absent.
    filename_stem : str
        Base filename (without extension) for the exported files.

    Returns
    -------
    tuple[Path, Path]
        Paths to the (JSON file, Markdown file) that were written.
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    ts: str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    json_path = out / f"{filename_stem}_{ts}.json"
    md_path = out / f"{filename_stem}_{ts}.md"

    json_path.write_text(json.dumps(history, indent=2, ensure_ascii=False), encoding="utf-8")

    md_lines: list[str] = [f"# Conversation Export — {ts}\n"]
    for turn in history:
        md_lines.append(f"## Turn {turn['turn']}\n")
        md_lines.append(f"**User:** {turn['user']}\n")
        md_lines.append(f"**Agent:** {turn['assistant']}\n")
        if turn.get("usage"):
            md_lines.append(f"*Tokens: {turn['usage']}*\n")
        md_lines.append("")

    md_path.write_text("\n".join(md_lines), encoding="utf-8")

    print(f"JSON saved : {json_path}")
    print(f"Markdown saved : {md_path}")
    return json_path, md_path


In [21]:
try:
    json_file, md_file = export_conversation(conversation_history, output_dir="exports")
except Exception as exc:
    print(f"Export failed: {exc}")


JSON saved : exports/conversation_20260527_081536.json
Markdown saved : exports/conversation_20260527_081536.md


In [22]:
!ls exports -lh

total 16K
-rwxrwxrwx 1 root root 7.9K May 27 08:15 conversation_20260527_081536.json
-rwxrwxrwx 1 root root 7.6K May 27 08:15 conversation_20260527_081536.md


## Interactive Chat Loop

A simple REPL loop to chat interactively with the agent in the notebook.
Type `exit` or `quit` to stop.

`KeyboardInterrupt` (Ctrl+C) is caught gracefully.


In [23]:
ANSI_RESET: str = "\033[0m"
ANSI_BLUE: str = "\033[1;34m"


def interactive_chat(
    client: AIProjectClient,
    agent_name: str,
) -> None:
    """Launch an interactive multi-turn chat session with the Foundry prompt agent.

    The session maintains a single conversation context.
    Type 'exit' or 'quit' to terminate the loop.
    Press Ctrl+C to interrupt gracefully.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the target agent.
    """
    openai_client = client.get_openai_client()

    try:
        conversation = openai_client.conversations.create()
    except Exception as exc:
        print(f"Failed to create conversation: {exc}")
        return

    print(f"Agent '{agent_name}' is ready. Type 'exit' or 'quit' to stop.")
    print("=" * 100)

    while True:
        try:
            user_input: str = input("You: ").strip()
        except KeyboardInterrupt:
            print(f"\n{ANSI_RESET}Session interrupted.")
            break

        if not user_input:
            continue

        if user_input.lower() in {"exit", "quit"}:
            print("Session ended.")
            break

        try:
            response = openai_client.responses.create(
                conversation=conversation.id,
                extra_body={
                    "agent_reference": {
                        "name": agent_name,
                        "type": "agent_reference",
                    }
                },
                input=user_input,
            )
        except Exception as exc:
            print(f"API error: {exc}")
            continue

        print(f"{ANSI_BLUE}Agent: {response.output_text}{ANSI_RESET}\n")


In [24]:
interactive_chat(project_client, AGENT_NAME)

Agent 'mslearn-agent' is ready. Type 'exit' or 'quit' to stop.


You:  hello


Agent: Hello! How can I help?



You:  How to use Mistral?


Agent: To use **Mistral models in Microsoft Foundry**, the basic flow is:

1. **Deploy a Mistral model**
2. **Call the inference endpoint**
3. **Pass the deployment name in the `model` parameter**

## 1) Deploy a Mistral model

In the Foundry portal:

- Go to **Discover** > **Models**
- Find a **Mistral** model
- Select **Deploy**
- Choose default or custom settings
- After deployment completes, use the deployment in Playground or code

Important notes from Microsoft Learn:

- For **models from partners and community** (which includes several Mistral models), you may need to **subscribe via Azure Marketplace** before deployment.
- During inference, the **deployment name** is what you pass in the `model` field.

## 2) Pick a Mistral model

Microsoft Learn lists multiple Mistral options in Foundry, including:

- `Codestral-2501`
- `Ministral-3B`
- `Mistral-small-2503`
- `Mistral-medium-2505`
- `Mistral-Large-3` (preview)
- Some older/community Mistral and Mixtral variants

Capabilities v

You:  exit


Session ended.


## List Existing Agents

Retrieve all agents registered in the current Foundry project.


In [25]:
def list_agents(client: AIProjectClient) -> list[object]:
    """List all prompt agents registered in the Foundry project.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.

    Returns
    -------
    list[object]
        A list of agent objects from the Foundry Agent Service.

    Raises
    ------
    RuntimeError
        If the list API call fails.
    """
    try:
        return list(client.agents.list())
    except Exception as exc:
        raise RuntimeError(f"Failed to list agents: {exc}") from exc


In [26]:
try:
    all_agents = list_agents(project_client)
    print(f"Found {len(all_agents)} agent(s) in the project:\n")
    for i, ag in enumerate(all_agents, start=1):
        print(f"{i:02}. {ag.name:30}  (id: {ag.id})")
except RuntimeError as exc:
    print(exc)


Found 11 agent(s) in the project:

01. mslearn-agent                   (id: mslearn-agent)
02. data-analysis-orchestrator-a2a  (id: data-analysis-orchestrator-a2a)
03. report-writer-agent             (id: report-writer-agent)
04. visualization-agent             (id: visualization-agent)
05. data-analysis-orchestrator      (id: data-analysis-orchestrator)
06. data-analyst-agent              (id: data-analyst-agent)
07. AzurePricingAgent               (id: AzurePricingAgent)
08. AzureOpenAIPricingAgent         (id: AzureOpenAIPricingAgent)
09. summarizer-agent                (id: summarizer-agent)
10. earth-knowledge-agent           (id: earth-knowledge-agent)
11. EmployeeSearchAgent             (id: EmployeeSearchAgent)


## List All Versions of an Agent

Enumerate every version that has been created for a given agent name.
This is important because `create_version` can be called multiple times,
accumulating versions that are not visible from `list_agents` alone.


In [27]:
def list_agent_versions(
    client: AIProjectClient,
    agent_name: str,
) -> list[object]:
    """List all versions of a specific prompt agent.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the agent whose versions to enumerate.

    Returns
    -------
    list[object]
        A list of agent version objects, ordered as returned by the API.

    Raises
    ------
    RuntimeError
        If the API call fails.
    """
    try:
        return list(client.agents.list_versions(agent_name=agent_name))
    except Exception as exc:
        raise RuntimeError(
            f"Failed to list versions for agent '{agent_name}': {exc}"
        ) from exc


In [28]:
try:
    versions = list_agent_versions(project_client, AGENT_NAME)
    print(f"Found {len(versions)} version(s) for agent '{AGENT_NAME}':\n")
    for v in versions:
        print(f"- Version {v.version}  status={getattr(v, 'status', 'n/a')}")
except RuntimeError as exc:
    print(exc)


Found 1 version(s) for agent 'mslearn-agent':

- Version 1  status=n/a


## Retrieve Agent Details

Fetch the full definition of a specific agent by name.
When no version is specified, the latest version is resolved automatically.


In [29]:
def get_agent_details(
    client: AIProjectClient,
    agent_name: str,
    version: Optional[str] = None,
) -> Any:
    """Retrieve full details for an agent version.

    When no version is given, resolves to the agent's latest version so the
    returned object always carries `version`, `status`, and `definition`.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the agent to retrieve.
    version : Optional[str]
        A specific version string. When None, the latest version is returned.

    Returns
    -------
    Any
        The agent version object with its full definition.

    Raises
    ------
    RuntimeError
        If the API call fails.
    """
    try:
        if version is None:
            parent = client.agents.get(agent_name=agent_name)
            version = parent.versions["latest"]["version"]

        return client.agents.get_version(
            agent_name=agent_name,
            agent_version=version,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Failed to get details for agent '{agent_name}': {exc}"
        ) from exc


In [30]:
try:
    agent_details = get_agent_details(project_client, AGENT_NAME)
    print(f"Agent Details '{AGENT_NAME}':")
    print(f"  Name      : {agent_details.name}")
    print(f"  Version   : {agent_details.version}")
    print(f"  Status    : {agent_details['status']}")
    print(f"  Definition: {agent_details.definition}")
except RuntimeError as exc:
    print(exc)


Agent Details 'mslearn-agent':
  Name      : mslearn-agent
  Version   : 1
  Status    : active
  Definition: {'kind': 'prompt', 'model': 'gpt-5.4', 'instructions': "You are an expert assistant specializing in Azure cloud services and AI.\n\nFor every factual question about Azure, Microsoft Foundry, .NET, M365, or\nPower Platform, you MUST call the Microsoft Learn MCP tools to retrieve\nauthoritative documentation before answering. Never answer purely from your\nown knowledge for product facts.\n\nCite the Microsoft Learn URLs you used at the end of each answer under a\n'Sources' heading. If the documentation does not contain the answer, say so\nexplicitly.\n\nBe concise, accurate, and structured. Use bullet points when listing items.\n", 'tools': [{'type': 'mcp', 'server_label': 'microsoft_learn', 'server_url': 'https://learn.microsoft.com/api/mcp', 'allowed_tools': {'tool_names': ['microsoft_docs_search', 'microsoft_docs_fetch', 'microsoft_code_sample_search']}, 'require_approval': '

## Update an Agent

Create a new version of an existing agent with updated instructions.
**This version correctly re-attaches the existing tools** to prevent
silently dropping MCP or other tool configurations.
Previous versions remain accessible by their version ID.


In [31]:
def update_agent(
    client: AIProjectClient,
    agent_name: str,
    model: str,
    new_instructions: str,
    tools: Optional[Sequence[object]] = None,
) -> object:
    """Update a prompt agent by creating a new version with revised instructions.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the agent to update.
    model : str
        The deployed model identifier.
    new_instructions : str
        The updated system instructions for the new version.
    tools : Optional[Sequence[object]]
        Tools to attach to the new version. Pass the same tools as the
        original version to avoid silently losing tool configuration.
        When None, the new version will have no tools.

    Returns
    -------
    object
        The newly created agent version.

    Raises
    ------
    RuntimeError
        If the API call fails.
    """
    definition_kwargs: dict[str, object] = {
        "model": model,
        "instructions": new_instructions,
    }
    if tools:
        definition_kwargs["tools"] = list(tools)

    try:
        return client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(**definition_kwargs),
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to update agent '{agent_name}': {exc}") from exc


In [32]:
UPDATED_INSTRUCTIONS: str = """\
Always respond in French.
"""

try:
    updated_agent = update_agent(
        client=project_client,
        agent_name=AGENT_NAME,
        model=AGENT_MODEL,
        new_instructions=UPDATED_INSTRUCTIONS,
        tools=[mslearn_tool],  # re-attach tools explicitly
    )
    print(f"Agent updated successfully.")
    print(f"  Name    : {updated_agent.name}")
    print(f"  Version : {updated_agent.version}")
except RuntimeError as exc:
    print(exc)


Agent updated successfully.
  Name    : mslearn-agent
  Version : 2


## Chat with the Updated Agent

Verify the new instructions are in effect by sending a test message.


In [33]:
test_question: str = "What is a prompt agent in Microsoft Foundry?"

try:
    test_answer, test_usage = chat_single_turn(project_client, AGENT_NAME, test_question)
    display(Markdown(f"**User:** {test_question}\n\n**Agent:** {test_answer}"))
    if test_usage:
        print(f"\nToken usage: {test_usage}")
except RuntimeError as exc:
    print(exc)


**User:** What is a prompt agent in Microsoft Foundry?

**Agent:** Dans Microsoft Foundry, un **prompt agent** est un agent IA **défini de manière déclarative par configuration**, plutôt que par du code applicatif complexe.

En pratique, il combine :

- un **modèle** du catalogue Foundry,
- des **instructions**,
- des **outils**,
- et des **prompts en langage naturel**

pour piloter son comportement.

Microsoft le décrit comme un agent créé via la configuration — **instructions, choix du modèle et outils** — puis **Foundry Agent Service gère automatiquement l’orchestration et l’hébergement**.

### En bref
Un prompt agent, c’est :

- un **agent “no-code / low-code”**
- **entièrement configuré** dans le portail Foundry, via API ou SDK
- adapté aux **prototypes rapides**, outils internes, et cas simples à modérés
- généralement un **agent unique** plutôt qu’un workflow multi-agents

### Quand l’utiliser
C’est le bon choix si tu veux :

- créer un agent rapidement,
- définir son comportement avec des instructions,
- lui ajouter des outils comme recherche, mémoire, fichiers, etc.,
- **sans écrire toute la logique d’orchestration toi-même**.

### Différence avec les autres agents Foundry
Microsoft Foundry distingue 3 types :

- **Prompt agents** : configuration simple, agent unique
- **Workflow agents** : orchestration multi-étapes / multi-agents
- **Hosted agents** : agents codés sur mesure, déployés en conteneur

### Formulation simple
Si je devais le résumer en une phrase :

> Un **prompt agent** dans Microsoft Foundry est un agent IA configuré à partir d’un modèle, d’instructions et d’outils, dont l’exécution et l’hébergement sont pris en charge par Foundry.

Source officielle Microsoft Learn :  
- https://learn.microsoft.com/azure/foundry/agents/overview  
- https://learn.microsoft.com/azure/foundry/agents/quickstarts/prompt-agent

Si tu veux, je peux aussi te montrer **un exemple concret de prompt agent** ou **la différence entre prompt agent et hosted agent**.


Token usage: {'input_tokens': 15016, 'output_tokens': 524, 'total_tokens': 15540}


## Clean Up

Delete the agent from the Foundry project when it is no longer needed.


In [34]:
def delete_agent(
    client: AIProjectClient,
    agent_name: str,
) -> None:
    """Permanently delete a named agent and all its versions from the Foundry project.

    Parameters
    ----------
    client : AIProjectClient
        An authenticated Foundry project client.
    agent_name : str
        The name of the agent to delete.

    Raises
    ------
    RuntimeError
        If the delete API call fails.
    """
    try:
        client.agents.delete(agent_name=agent_name)
        print(f"Agent '{agent_name}' deleted successfully.")
    except Exception as exc:
        raise RuntimeError(f"Failed to delete agent '{agent_name}': {exc}") from exc

In [35]:
print(f"Agent to delete: '{AGENT_NAME}'")

Agent to delete: 'mslearn-agent'


In [36]:
try:
    delete_agent(project_client, AGENT_NAME)
except RuntimeError as exc:
    print(exc)

Agent 'mslearn-agent' deleted successfully.


In [37]:
try:
    all_agents = list_agents(project_client)
    print(f"Found {len(all_agents)} agent(s) in the project:\n")
    for i, ag in enumerate(all_agents, start=1):
        print(f"  {i:02}. {ag.name:30}  (id: {ag.id})")
except RuntimeError as exc:
    print(exc)

Found 10 agent(s) in the project:

  01. data-analysis-orchestrator-a2a  (id: data-analysis-orchestrator-a2a)
  02. report-writer-agent             (id: report-writer-agent)
  03. visualization-agent             (id: visualization-agent)
  04. data-analysis-orchestrator      (id: data-analysis-orchestrator)
  05. data-analyst-agent              (id: data-analyst-agent)
  06. AzurePricingAgent               (id: AzurePricingAgent)
  07. AzureOpenAIPricingAgent         (id: AzureOpenAIPricingAgent)
  08. summarizer-agent                (id: summarizer-agent)
  09. earth-knowledge-agent           (id: earth-knowledge-agent)
  10. EmployeeSearchAgent             (id: EmployeeSearchAgent)
